In [1]:
import pandas as pd
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score
import numpy as np

# Player stats

# Players and models

In [2]:
players_df = pd.read_csv('stored_datasets/All Pitchers Cleaned.csv')
results_logreg = pd.read_csv('stored_results/Results_Logreg_No_GS.csv')
results_dtree = pd.read_csv('stored_results/Results_DTree_GS_Best_F1_Stratified_KFold.csv')
results_rfc = pd.read_csv('stored_results/Results_RFC_GS_Best_Recall_Stratified_KFold.csv')
results_etc = pd.read_csv('stored_results/Results_ETC_GS_Best_Recall_Standard_KFold.csv')
results_gbc = pd.read_csv('stored_results/Results_GBC_GS_Best_Recall_Standard_KFold.csv')
results_svc = pd.read_csv('stored_results/Results_SVC_no_GS_Stratified_KFold.csv')
results_nn = pd.read_csv('stored_results/Results_Neural_Network_1.csv')

In [3]:
results_logreg

,Player,Hall of Fame,Prediction,Prediction Correct,Prediction Confidence
0,Cy Young,1,1,1,0.999999
1,Pud Galvin,1,1,1,0.618978
2,Walter Johnson,1,1,1,0.999999
3,Phil Niekro,1,1,1,0.918764
4,Nolan Ryan,1,1,1,0.998704
...,...,...,...,...,...
1210,Charlie Robertson,0,0,1,0.999562
1211,Sammy Ellis,0,0,1,0.993309
1212,Lil Stoner,0,0,1,0.999577
1213,Johnny Humphries,0,0,1,0.999019


In [4]:
player_models = pd.DataFrame()
player_models['Player'] = players_df['Player']
player_models['Hall of Fame'] = players_df['Hall of Fame']

player_models['Logreg Pred'] = results_logreg['Prediction']
player_models['DTree Pred'] = results_dtree['Prediction']
player_models['RFC Pred'] = results_rfc['Prediction']
player_models['ETC Pred'] = results_etc['Prediction']
player_models['GBC Pred'] = results_gbc['Prediction']
player_models['SVC Pred'] = results_svc['Prediction']

pred_cols = ['Logreg Pred', 'DTree Pred', 'RFC Pred', 'ETC Pred', 'GBC Pred', 'SVC Pred']
player_models['Correct Models (out of 6)'] = player_models.apply(
    lambda row: sum(row[col] == row["Hall of Fame"] for col in pred_cols),
    axis=1
)

player_models['Mean Confidence in Prediction'] = (results_logreg['Prediction Confidence'] + results_dtree['Prediction Confidence'] + results_rfc['Prediction Confidence'] + results_etc['Prediction Confidence'] + results_gbc['Prediction Confidence'] + results_svc['Prediction Confidence']) / 6

player_models

,Player,Hall of Fame,Logreg Pred,DTree Pred,RFC Pred,ETC Pred,GBC Pred,SVC Pred,Correct Models (out of 6),Mean Confidence in Prediction
0,Cy Young,1,1,1,1,1,1,1,6,0.966098
1,Pud Galvin,1,1,1,1,1,1,1,6,0.847402
2,Walter Johnson,1,1,1,1,1,1,1,6,0.982103
3,Phil Niekro,1,1,1,1,1,1,1,6,0.931801
4,Nolan Ryan,1,1,1,1,1,1,1,6,0.925245
...,...,...,...,...,...,...,...,...,...,...
1210,Charlie Robertson,0,0,0,0,0,0,0,6,0.992825
1211,Sammy Ellis,0,0,0,0,0,0,0,6,0.991213
1212,Lil Stoner,0,0,0,0,0,0,0,6,0.993293
1213,Johnny Humphries,0,0,0,0,0,0,0,6,0.992819


In [18]:
player_models['Correct Models (out of 6)'].value_counts()

Correct Models (out of 6)
6    1105
5      41
4      21
0      16
2      14
1      10
3       8
Name: count, dtype: int64

In [18]:
dfs = [
    results_logreg,
    results_dtree,
    results_rfc,
    results_etc,
    results_gbc,
    results_svc
]

alpha = 2
beta = 2

for df in dfs:
    
    # Recreate Prediction Correct EXACTLY like before
    if {'Hall of Fame', 'Prediction'}.issubset(df.columns):
        df['Prediction Correct'] = (df['Hall of Fame'] == df['Prediction'])

    # Now calculate adjusted score (will behave correctly)
    df['Adjusted Prediction Score'] = np.where(
        df['Prediction Correct'],
        df['Prediction Confidence'] ** alpha,
        -beta * (df['Prediction Confidence'] ** alpha)
    )


In [21]:
player_models['Mean Adjusted Prediction Score'] = (results_logreg['Adjusted Prediction Score'] + results_dtree['Adjusted Prediction Score'] + results_rfc['Adjusted Prediction Score'] + results_etc['Adjusted Prediction Score'] + results_gbc['Adjusted Prediction Score'] + results_svc['Adjusted Prediction Score']) / 6
player_models

,Player,Hall of Fame,Logreg Pred,DTree Pred,RFC Pred,ETC Pred,GBC Pred,SVC Pred,Correct Models (out of 6),Mean Confidence in Prediction,Mean Adjusted Prediction Score
0,Cy Young,1,1,1,1,1,1,1,6,0.966098,0.935399
1,Pud Galvin,1,1,1,1,1,1,1,6,0.847402,0.739864
2,Walter Johnson,1,1,1,1,1,1,1,6,0.982103,0.965164
3,Phil Niekro,1,1,1,1,1,1,1,6,0.931801,0.873134
4,Nolan Ryan,1,1,1,1,1,1,1,6,0.925245,0.862487
...,...,...,...,...,...,...,...,...,...,...,...
1210,Charlie Robertson,0,0,0,0,0,0,0,6,0.992825,0.985939
1211,Sammy Ellis,0,0,0,0,0,0,0,6,0.991213,0.982775
1212,Lil Stoner,0,0,0,0,0,0,0,6,0.993293,0.986838
1213,Johnny Humphries,0,0,0,0,0,0,0,6,0.992819,0.985920


In [ ]:
# Calculate adjusted score
    df['Adjusted Prediction Score'] = np.where(
        df['Prediction Correct'],
        df['Prediction Confidence'] ** alpha,
        -beta * (df['Prediction Confidence'] ** alpha)
    )

In [5]:
import numpy as np

import numpy as np

def add_adjusted_score_inplace(df, alpha=1, beta=1):
    # Verify expected columns exist
    if 'Prediction Correct' not in df.columns:
        raise KeyError("Column 'Prediction Correct' not found in dataframe.")
    if 'Prediction Confidence' not in df.columns:
        raise KeyError("Column 'Prediction Confidence' not found in dataframe.")

    # Add column IN PLACE
    df.loc[:, 'Adjusted Prediction Score'] = np.where(
        df['Prediction Correct'],
        df['Prediction Confidence'] ** alpha,
        -beta * (df['Prediction Confidence'] ** alpha)
    )

In [7]:
dfs = [
    results_logreg,
    results_dtree,
    results_rfc,
    results_etc,
    results_gbc,
    results_svc
]

for df in dfs:
    add_adjusted_score_inplace(df, alpha=1, beta=1)


In [8]:
results_logreg

,Player,Hall of Fame,Prediction,Prediction Correct,Prediction Confidence,Adjusted Prediction Score
0,Cy Young,1,1,1,0.999999,0.999999
1,Pud Galvin,1,1,1,0.618978,0.618978
2,Walter Johnson,1,1,1,0.999999,0.999999
3,Phil Niekro,1,1,1,0.918764,0.918764
4,Nolan Ryan,1,1,1,0.998704,0.998704
...,...,...,...,...,...,...
1210,Charlie Robertson,0,0,1,0.999562,0.999562
1211,Sammy Ellis,0,0,1,0.993309,0.993309
1212,Lil Stoner,0,0,1,0.999577,0.999577
1213,Johnny Humphries,0,0,1,0.999019,0.999019


In [24]:
results_logreg = add_adjusted_score(results_logreg, alpha=1, beta=1)
results_dtree  = add_adjusted_score(results_dtree,  alpha=1, beta=1)
results_rfc    = add_adjusted_score(results_rfc,    alpha=1, beta=1)
results_etc    = add_adjusted_score(results_etc,    alpha=1, beta=1)
results_gbc    = add_adjusted_score(results_gbc,    alpha=1, beta=1)
results_svc    = add_adjusted_score(results_svc,    alpha=1, beta=1)

In [5]:
player_models['Mean Adjusted Prediction Score'] = (results_logreg['Adjusted Prediction Score'] + results_dtree['Adjusted Prediction Score'] + results_rfc['Adjusted Prediction Score'] + results_etc['Adjusted Prediction Score'] + results_gbc['Adjusted Prediction Score'] + results_svc['Adjusted Prediction Score']) / 6
player_models

KeyError: 'Adjusted Prediction Score'

# Model Comparison

In [5]:
# Subject to change if more models are tried later. Refer to Part 4a for full list.
model_files = {
    "Logistic Regression": "stored_results/Results_Logreg_No_GS.csv",
    "Decision Tree (GS for F1, Stratified KFold)": "stored_results/Results_DTree_GS_Best_F1_Stratified_KFold.csv",
    "Random Forest (GS for Recall, Stratified KFold)": "stored_results/Results_RFC_GS_Best_Recall_Stratified_KFold.csv",
    "Extra Trees (GS for Recall, Standard KFold)": "stored_results/Results_ETC_GS_Best_Recall_Standard_KFold.csv",
    "Gradient Booster (GS for Recall, Standard KFold)": "stored_results/Results_GBC_GS_Best_Recall_Standard_KFold.csv",
    "Support Vector (no GS, Stratified KFold)": "stored_results/Results_SVC_no_GS_Stratified_KFold.csv",
    "Neural Network": "stored_results/Results_Neural_Network_1.csv",
}

In [6]:
import pandas as pd
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score, accuracy_score
)
import pandas as pd
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score, accuracy_score, roc_auc_score
)

model_stats = []



# Define penalty weights
alpha, beta = 2, 2




for model_name, file_path in model_files.items():
    df = pd.read_csv(file_path)
    y_true = df['Hall of Fame']
    y_pred = df['Prediction']

    # Confirm binary labels
    labels = sorted(list(set(y_true) | set(y_pred)))
    print(f"{model_name} labels: {labels}")  # sanity check

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    # Optional: if you have probabilities
    y_proba = df['Prediction Confidence'] if 'y_proba' in df.columns else None


    # Use probabilities if available
    if y_proba is not None:
        roc_auc = roc_auc_score(y_true, y_proba)
    else:
        roc_auc = roc_auc_score(y_true, y_pred)

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    recall = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0)




    # Compute adjusted prediction score if probabilities are available

    
    if 'Prediction Confidence' in df.columns:
        p = df['Prediction Confidence']
        df['Adjusted Prediction Score'] = np.where(
            df['Prediction Correct'],
            p**alpha,
            -beta * p**alpha
        )
        mean_adj_score = df['Adjusted Prediction Score'].mean()
    else:
        mean_adj_score = np.nan  # no confidence info




    

    model_stats.append({
        "Model": model_name,
        "True Negatives": tn,
        "False Positives": fp,
        "False Negatives": fn,
        "True Positives": tp,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC AUC": roc_auc,
        "Mean Adjusted Prediction Score": mean_adj_score
    })


results_df_unweighed = pd.DataFrame(model_stats)

results_df_unweighed

Logistic Regression labels: [0, 1]
Decision Tree (GS for F1, Stratified KFold) labels: [0, 1]
Random Forest (GS for Recall, Stratified KFold) labels: [0, 1]
Extra Trees (GS for Recall, Standard KFold) labels: [0, 1]
Gradient Booster (GS for Recall, Standard KFold) labels: [0, 1]
Support Vector (no GS, Stratified KFold) labels: [0, 1]
Neural Network labels: [0, 1]


,Model,True Negatives,False Positives,False Negatives,True Positives,Accuracy,Precision,Recall,F1 Score,ROC AUC,Mean Adjusted Prediction Score
0,Logistic Regression,1117,16,34,48,0.958848,0.750000,0.585366,0.657534,0.785622,0.868560
1,"Decision Tree (GS for F1, Stratified KFold)",1101,32,32,50,0.947325,0.609756,0.609756,0.609756,0.790756,0.837889
2,"Random Forest (GS for Recall, Stratified KFold)",1123,10,31,51,0.966255,0.836066,0.621951,0.713287,0.806563,0.873379
3,"Extra Trees (GS for Recall, Standard KFold)",1110,23,22,60,0.962963,0.722892,0.731707,0.727273,0.855704,0.849992
4,"Gradient Booster (GS for Recall, Standard KFold)",1114,19,29,53,0.960494,0.736111,0.646341,0.688312,0.814786,0.884231
5,"Support Vector (no GS, Stratified KFold)",1126,7,54,28,0.949794,0.800000,0.341463,0.478632,0.667643,0.789771
6,Neural Network,1117,16,26,56,0.965432,0.777778,0.682927,0.727273,0.834403,0.018855
